# Lab 1: CNN Robustness Report - Student Notebook

## Objective
Build a comprehensive CNN robustness report by:
1. Training a baseline CNN model
2. Detecting overfitting from training curves
3. Applying Dropout regularization and comparing against the baseline
4. Running adversarial attacks (FGSM)
5. Generating a final robustness report

## Deliverables
Your final report must include:
- ✅ Clean accuracy (baseline model)
- ✅ Noisy accuracy (Gaussian noise perturbation)
- ✅ Adversarial accuracy (FGSM attack)
- ✅ Model size (MB)
- ✅ Inference time (ms per image)
- ✅ Observations (overfitting analysis, regularization effectiveness, robustness assessment)

## Instructions
- **🔴 RED CODE**: Code you need to complete (fill in the blanks)
- **🟢 GREEN CODE**: Complete code already provided
- **📝 COMMENTS**: Explain what each section does
- **💡 HINTS**: Refer to Part 1, 2, 3 notebooks from class

## Section 1: Setup & Data Loading

In [1]:
# 🟢 GREEN CODE - Complete (Run this as-is)
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18
from tqdm import tqdm

np.random.seed(42)
torch.manual_seed(42)

print("✅ Libraries imported")

✅ Libraries imported


In [2]:
# 🟢 GREEN CODE - Device and paths
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DATASET_PATH = r"C:\Users\Lucifer\OneDrive\BITS\AI_Quality_Engineering\dataset"
TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VAL_PATH = os.path.join(DATASET_PATH, "val")
TEST_PATH = os.path.join(DATASET_PATH, "test")

Device: cuda


In [3]:
# 🟢 GREEN CODE - Data transformations and loading
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(TRAIN_PATH, transform=transform)
val_dataset = ImageFolder(VAL_PATH, transform=transform)
test_dataset = ImageFolder(TEST_PATH, transform=transform)

class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Classes: {class_names}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Classes: ['animal', 'name_board', 'pedestrian', 'pothole', 'road_sign', 'speed_breaker', 'vehicle']
Train: 796 | Val: 234 | Test: 117


In [4]:
# 🔴 RED CODE - TODO: Create data loaders
# Create train_loader, val_loader, test_loader with batch_size=64
# Use shuffle=True for train, shuffle=False for val/test

BATCH_SIZE = 64
train_loader = DataLoader(  # TODO: Complete this line
    
)
val_loader = DataLoader(  # TODO: Complete this line
    
)
test_loader = DataLoader(  # TODO: Complete this line
    
)

print(f"✅ Data loaders created with batch size {BATCH_SIZE}")

TypeError: DataLoader.__init__() missing 1 required positional argument: 'dataset'

## Section 2: Define Models with Regularization

In [ ]:
# 🟢 GREEN CODE - Baseline model (no regularization, BatchNorm stripped)
#
# Why strip BatchNorm: BN is itself a strong implicit regularizer. Leaving it
# in means the "unregularized" baseline is partially regularized, so the gap
# vs the Dropout model becomes too small to see. Removing BN exposes the raw
# over-parameterized ResNet so it overfits visibly on small data.
def _replace_bn_with_identity(module):
    """Recursively swap every nn.BatchNorm2d in `module` with nn.Identity."""
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name, nn.Identity())
        else:
            _replace_bn_with_identity(child)


class BaselineModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.resnet = resnet18(pretrained=False)
        _replace_bn_with_identity(self.resnet)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

print("✅ BaselineModel defined (BatchNorm stripped)")

In [ ]:
# 🔴 RED CODE - TODO: Create Dropout Model (BN stripped, deep dropout)
#
# Goal: regularize the high-capacity feature layers (layer3, layer4) and the
# final classifier. Dropout2d drops whole channels, which is the correct
# form for convolutional feature maps. Don't put Dropout2d in the early
# stages — on a small dataset that destroys low-level features and makes
# training accuracy jump around epoch-to-epoch.

class DropoutModel(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.3):
        super().__init__()
        base = resnet18(pretrained=False)
        _replace_bn_with_identity(base)  # reuse helper from BaselineModel cell

        # 🟢 GREEN — stem and early stages are kept clean (no dropout)
        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2

        # 🔴 TODO: Wrap base.layer3 and base.layer4 in nn.Sequential, adding
        #          nn.Dropout2d(dropout_rate) after each. Example pattern:
        #              nn.Sequential(base.layerN, nn.Dropout2d(dropout_rate))
        self.layer3 = ___  # TODO
        self.layer4 = ___  # TODO

        self.avgpool = base.avgpool

        # 🔴 TODO: Build the FC block as nn.Sequential of:
        #            nn.Flatten(), nn.Dropout(0.5), nn.Linear(in_features, num_classes)
        in_features = base.fc.in_features
        self.fc = ___  # TODO

    def forward(self, x):
        # 🔴 TODO: Implement the forward pass through:
        #          stem -> layer1 -> layer2 -> layer3 -> layer4 -> avgpool -> fc
        pass

print("✅ DropoutModel defined")

## Section 3: Training & Evaluation Functions

In [ ]:
# 🟢 GREEN CODE - Training function (complete)
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("✅ train_epoch() defined")

In [ ]:
# 🔴 RED CODE - TODO: Complete evaluation function
def evaluate(model, loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            # TODO: Forward pass
            outputs = ___
            # TODO: Calculate loss
            loss = ___
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("✅ evaluate() defined")

## Section 4: Train All Three Models

In [ ]:
# 🔴 RED CODE - TODO: Train Baseline Model
# Create model, optimizer, criterion and train for 20 epochs

print("\n" + "="*60)
print("Training BASELINE Model (No Regularization)")
print("="*60)

baseline_model = BaselineModel(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_baseline = optim.SGD(baseline_model.parameters(), lr=5e-3)  # TODO: Add weight_decay=0 (or no weight decay)

baseline_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
NUM_EPOCHS = 20

# TODO: Complete the training loop
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(baseline_model, train_loader, criterion, optimizer_baseline, device)
    val_loss, val_acc = evaluate(baseline_model, val_loader, criterion, device)
    
    baseline_history['train_loss'].append(train_loss)
    baseline_history['train_acc'].append(train_acc)
    baseline_history['val_loss'].append(val_loss)
    baseline_history['val_acc'].append(val_acc)
    
    # TODO: Print epoch results (Epoch [X/20] Train: XXX% Val: XXX%)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("✅ Baseline model trained")

In [ ]:
# 🔴 RED CODE - TODO: Train Dropout Model
print("\n" + "="*60)
print("Training DROPOUT Model (Dropout Rate = 0.5)")
print("="*60)

dropout_model = DropoutModel(num_classes, dropout_rate=0.5).to(device)
optimizer_dropout = optim.SGD(dropout_model.parameters(), lr=5e-3)

dropout_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

# TODO: Complete the training loop
for epoch in range(NUM_EPOCHS):
    # TODO: Implement (same pattern as above)
    pass

print("✅ Dropout model trained")

## Section 5: Detect Overfitting

In [ ]:
# 🔴 RED CODE - TODO: Plot training curves and analyze overfitting
# Create a figure with 2 subplots (one for each model)
# Each subplot should show train and val accuracy curves

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Overfitting Analysis: Baseline vs Dropout', fontsize=16, fontweight='bold')

models_data = [
    ('Baseline', baseline_history, axes[0]),
    ('Dropout', dropout_history, axes[1])
]

# TODO: Complete the plotting loop
for model_name, history, ax in models_data:
    # TODO: Plot train and val accuracy
    # TODO: Set labels, title, legend
    # TODO: Add grid
    pass

plt.tight_layout()
plt.savefig('overfitting_comparison.png', dpi=150)
plt.show()

print("✅ Overfit curves plotted")

In [ ]:
# 🔴 RED CODE - TODO: Calculate overfitting metrics
# For each model, calculate the gap between final train and val accuracy

overfitting_analysis = {}

# Baseline
baseline_gap = baseline_history['train_acc'][-1] - baseline_history['val_acc'][-1]
overfitting_analysis['Baseline'] = baseline_gap

# Dropout
# TODO: Calculate Dropout gap
dropout_gap = ___
overfitting_analysis['Dropout'] = dropout_gap

print("\n📊 OVERFITTING ANALYSIS (Train-Val Gap):")
for model_name, gap in overfitting_analysis.items():
    print(f"{model_name:20s}: {gap:6.2f}% gap", end="")
    if gap > 10:
        print(" ⚠️  HIGH OVERFITTING")
    elif gap > 5:
        print(" ⚠️  MODERATE OVERFITTING")
    else:
        print(" ✅ GOOD GENERALIZATION")

## Section 6: Model Complexity Analysis

In [ ]:
# 🟢 GREEN CODE - Helper functions
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    torch.save(model.state_dict(), "temp_model.pth")
    size_mb = os.path.getsize("temp_model.pth") / (1024 * 1024)
    os.remove("temp_model.pth")
    return size_mb

print("✅ Helper functions defined")

In [ ]:
# 🔴 RED CODE - TODO: Calculate model sizes and parameters
model_complexity = {}

# TODO: For each model, calculate parameters and size
for model_name, model in [('Baseline', baseline_model), ('Dropout', dropout_model)]:
    params = count_parameters(model)
    size = get_model_size_mb(model)
    model_complexity[model_name] = {'params': params, 'size': size}
    print(f"{model_name}: {params:,} parameters | {size:.2f} MB")

## Section 7: Robustness Testing - Adversarial Attacks

In [ ]:
# 🔴 RED CODE - TODO: Implement Gaussian Noise Perturbation
def add_gaussian_noise(images, noise_std=0.1):
    """
    Add Gaussian noise to images.
    
    Args:
        images: Tensor of shape (B, C, H, W)
        noise_std: Standard deviation of Gaussian noise
    
    Returns:
        Noisy images (clipped to valid range [-1, 1])
    """
    # TODO: Create random noise with torch.randn_like
    # TODO: Add noise to images
    # TODO: Clip to [-1, 1]
    noise = ___
    noisy_images = ___
    return torch.clamp(noisy_images, -1, 1)

print("✅ Gaussian noise function defined")

In [ ]:
# 🔴 RED CODE - TODO: Implement FGSM Attack
def fgsm_attack(model, images, labels, device, epsilon=0.05):
    """
    Fast Gradient Sign Method (FGSM) Attack.
    
    Perturb images in the direction of the gradient to fool the model.
    
    Args:
        model: Neural network
        images: Input images
        labels: True labels
        device: Device to run on
        epsilon: Attack strength
    
    Returns:
        Adversarial images
    """
    # TODO: Enable gradient tracking for images
    images.requires_grad = ___
    
    # Forward pass
    outputs = model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels)
    
    # TODO: Compute gradients
    model.zero_grad()
    loss.backward()
    
    # TODO: Get gradient sign and create adversarial examples
    data_grad = images.grad.data
    sign_data_grad = ___
    
    # TODO: Create perturbed images
    perturbed_images = ___
    
    # Clip to valid range
    return torch.clamp(perturbed_images, -1, 1).detach()

print("✅ FGSM attack function defined")

In [ ]:
# 🔴 RED CODE - TODO: Evaluate robustness on noisy images
def evaluate_on_noisy(model, loader, device, noise_std=0.1):
    """Evaluate model on Gaussian noise-perturbed images"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Testing noise (σ={noise_std})", leave=False):
            images, labels = images.to(device), labels.to(device)
            # TODO: Add Gaussian noise
            noisy_images = ___
            # TODO: Get predictions
            outputs = ___
            _, predicted = torch.max(outputs, 1)
            # TODO: Update accuracy
            total += ___
            correct += ___
    
    accuracy = 100 * correct / total
    return accuracy

print("✅ Noisy evaluation function defined")

In [ ]:
# 🔴 RED CODE - TODO: Evaluate robustness against FGSM attacks
def evaluate_on_adversarial(model, loader, device, epsilon=0.05):
    """Evaluate model on FGSM adversarial examples"""
    model.eval()
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc=f"Testing FGSM (ε={epsilon})", leave=False):
        images, labels = images.to(device), labels.to(device)
        # TODO: Generate adversarial examples
        adv_images = ___
        # TODO: Evaluate on adversarial images
        # TODO: Update accuracy
        pass
    
    accuracy = 100 * correct / total
    return accuracy

print("✅ Adversarial evaluation function defined")

In [ ]:
# 🔴 RED CODE - TODO: Test all models on clean, noisy, and adversarial data
print("\n" + "="*70)
print("ROBUSTNESS EVALUATION")
print("="*70)

robustness_results = {}

for model_name, model in [('Baseline', baseline_model), ('Dropout', dropout_model)]:
    print(f"\n{model_name}:")
    
    # Clean accuracy
    clean_loss, clean_acc = evaluate(model, test_loader, criterion, device)
    # TODO: Get noisy accuracy
    noisy_acc = ___
    # TODO: Get adversarial accuracy
    adv_acc = ___
    
    robustness_results[model_name] = {
        'clean': clean_acc,
        'noisy': noisy_acc,
        'adversarial': adv_acc
    }
    
    print(f"  Clean Accuracy:       {clean_acc:.2f}%")
    print(f"  Noisy Accuracy (σ=0.1): {noisy_acc:.2f}%")
    print(f"  Adversarial Accuracy (ε=0.05): {adv_acc:.2f}%")

## Section 8: Generate Final Report

In [ ]:
# 🔴 RED CODE - TODO: Create comprehensive robustness report
print("\n" + "="*80)
print("🎯 CNN ROBUSTNESS REPORT")
print("="*80)

print("\n📊 ACCURACY METRICS:")
print("-" * 80)
print(f"{'Model':<25} {'Clean':<15} {'Noisy':<15} {'Adversarial':<15}")
print("-" * 80)

# TODO: Print results for each model
for model_name in ['Baseline', 'Dropout']:
    # TODO: Get results and print in table format
    pass

print("\n⏱️  INFERENCE TIME & MODEL SIZE:")
print("-" * 80)
# TODO: Print inference time and model size for each model

In [ ]:
# 🔴 RED CODE - TODO: Inference time analysis
print("\n📏 MODEL COMPLEXITY:")
print("-" * 80)
print(f"{'Model':<25} {'Parameters':<20} {'Size (MB)':<15}")
print("-" * 80)

# TODO: Print parameters and size for each model
for model_name in ['Baseline', 'Dropout']:
    params = model_complexity[model_name]['params']
    size = model_complexity[model_name]['size']
    # TODO: Print in table format
    pass

In [ ]:
# 🔴 RED CODE - TODO: Write your observations
print("\n💡 OBSERVATIONS & ANALYSIS:")
print("="*80)

observations = """
Please answer the following questions:

1. OVERFITTING ANALYSIS:
   - Which model shows the least overfitting? Why?
   - TODO: Write your observation here
   
2. REGULARIZATION EFFECTIVENESS:
   - How much does Dropout reduce the train-val gap compared to the Baseline?
   - Why does deep Dropout2d (after layer3/layer4) work better than dropout
     placed only at the final FC layer?
   - TODO: Write your observation here
   
3. ROBUSTNESS ASSESSMENT:
   - Which model is most robust to adversarial attacks?
   - What's your hypothesis for why?
   - TODO: Write your observation here
   
4. ACCURACY-COMPLEXITY TRADEOFF:
   - Is there a correlation between model size and accuracy?
   - TODO: Write your observation here
   
5. RECOMMENDATIONS:
   - Which model would you deploy in production and why?
   - TODO: Write your recommendation here
"""

print(observations)

# TODO: Write your answers below
YOUR_OBSERVATIONS = """
"""

print(YOUR_OBSERVATIONS)

In [ ]:
# 🔴 RED CODE - TODO: Create final summary visualization
# Create a bar chart comparing clean, noisy, and adversarial accuracy for both models

fig, ax = plt.subplots(figsize=(10, 6))

# TODO: Prepare data for plotting
# Hint: Use model names ['Baseline', 'Dropout'] as x-axis, and plot 3 bars per model

# TODO: Create grouped bar chart
# Note: You'll need to offset the bars for each accuracy type

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Robustness Comparison: Clean vs Noisy vs Adversarial', fontsize=14, fontweight='bold')
ax.legend(['Clean', 'Noisy', 'Adversarial'], fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('robustness_summary.png', dpi=150)
plt.show()

print("✅ Summary visualization created")

## Submission Checklist

Before submitting, ensure your report includes:

- [ ] **Clean Accuracy**: Baseline and Dropout model test accuracy
- [ ] **Noisy Accuracy**: Accuracy with Gaussian noise (σ=0.1)
- [ ] **Adversarial Accuracy**: Accuracy with FGSM attack (ε=0.05)
- [ ] **Model Size**: Size in MB for both models
- [ ] **Inference Time**: Per-image inference time
- [ ] **Overfitting Analysis**: Explanation of train-val gap for each model
- [ ] **Regularization Analysis**: How much did Dropout reduce overfitting and why?
- [ ] **Robustness Assessment**: Which model is most robust and why?
- [ ] **Visualizations**: Training curves and robustness comparison chart
- [ ] **Observations**: Detailed answers to all 5 questions above

## Questions & Tips

- If stuck, refer to Part 1, 2, and 3 notebooks from class
- Look for `# TODO:` comments to find missing code
- Use the solution notebook only as a last resort!
- Good luck! 🚀